# Ejercicios 4 y 5 · Model Card + Datasheet — Proyecto: deserción estudiantil (UCI #697)

**Qué hace este cuaderno:** reproduce los Ejercicios 4, 5 y 6 del cuaderno de la Dra. Gaur (Session 7) sobre **otro dataset**: *Predict Students' Dropout and Academic Success* (UCI ID 697, CC BY 4.0), descrito en Realinho et al. (2022), *Data* 7(11):146, DOI 10.3390/data7110146.

**Cómo usarlo (Colab):** ejecuta las celdas en orden. Todos los números del Model Card (accuracy, macro-F1, recall por clase, métricas por subgrupo) se calculan en la corrida. Los datos del Datasheet vienen de Realinho et al. (2022) y de la página de UCI; lo que **no está en esas fuentes** queda como `UNKNOWN — to investigate`.


In [1]:
%pip -q install ucimlrepo
import numpy as np, pandas as pd, sklearn, platform
from datetime import date
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.pipeline import make_pipeline
from sklearn.ensemble import GradientBoostingClassifier
from sklearn.dummy import DummyClassifier
from sklearn.metrics import accuracy_score, f1_score, recall_score
from ucimlrepo import fetch_ucirepo

SEEDS = range(5)          # mismas 5 semillas que en la clase
TEST_SIZE = 0.20          # partición 80/20 recomendada por UCI para este dataset

In [2]:
# --- Carga del dataset (UCI #697) ---
ds = fetch_ucirepo(id=697)
X = ds.data.features.copy()
y_raw = ds.data.targets.iloc[:, 0]

le = LabelEncoder()
y = le.fit_transform(y_raw)
CLASSES = list(le.classes_)
N, P = X.shape
counts = y_raw.value_counts()
class_counts_txt = ", ".join(f"{k}: {v} ({v/N:.1%})" for k, v in counts.items())

# Variables que describen el rendimiento en semestres ya cursados (posible problema de
# disponibilidad temporal si el uso previsto es predecir AL MOMENTO DE LA MATRÍCULA).
sem_cols = [c for c in X.columns if "curricular units" in c.lower()]

print("Filas, columnas:", N, P)
print("Clases:", CLASSES)
print("Distribución:", class_counts_txt)
print("Columnas de unidades curriculares (semestres cursados):", len(sem_cols))
print("Valores faltantes totales en X:", int(X.isna().sum().sum()))

# --- Verificación contra Realinho et al. (2022), Data 7(11):146 ---
PAPER = {"records": 4424, "attributes_incl_target": 35,
         "classes": {"Graduate": 2209, "Dropout": 1421, "Enrolled": 794}}
obs_classes = {str(k): int(v) for k, v in counts.items()}
check_records = (N == PAPER["records"])
check_classes = (obs_classes == PAPER["classes"])
check_attrs = ((P + 1) == PAPER["attributes_incl_target"])
paper_check_txt = (f"records {N} vs paper {PAPER['records']}: {'OK' if check_records else 'MISMATCH'}; "
                   f"class counts {'OK' if check_classes else 'MISMATCH'}; "
                   f"attributes incl. target {P+1} vs paper {PAPER['attributes_incl_target']}: "
                   f"{'OK' if check_attrs else 'MISMATCH (UCI lists 36 features, paper 34 predictors + target)'}")
print("Contraste con el artículo:", paper_check_txt)

Filas, columnas: 4424 36
Clases: ['Dropout', 'Enrolled', 'Graduate']
Distribución: Graduate: 2209 (49.9%), Dropout: 1421 (32.1%), Enrolled: 794 (17.9%)
Columnas de unidades curriculares (semestres cursados): 12
Valores faltantes totales en X: 0
Contraste con el artículo: records 4424 vs paper 4424: OK; class counts OK; attributes incl. target 37 vs paper 35: MISMATCH (UCI lists 36 features, paper 34 predictors + target)


In [3]:
# --- Experimento: 5 semillas, partición estratificada, StandardScaler SOLO con train ---
accs, f1s, majs, recalls = [], [], [], []
sub_recalls = {}     # recall de la clase 'Dropout' por subgrupo de género (si existe la columna)
has_gender = "Gender" in X.columns
drop_idx = CLASSES.index("Dropout")

for s in SEEDS:
    Xtr, Xte, ytr, yte = train_test_split(X, y, test_size=TEST_SIZE, stratify=y, random_state=s)
    model = make_pipeline(StandardScaler(),
                          GradientBoostingClassifier(n_estimators=200, max_depth=3, random_state=s))
    model.fit(Xtr, ytr)
    pred = model.predict(Xte)
    accs.append(accuracy_score(yte, pred))
    f1s.append(f1_score(yte, pred, average="macro"))
    recalls.append(recall_score(yte, pred, average=None, labels=list(range(len(CLASSES)))))
    majs.append(accuracy_score(yte, DummyClassifier(strategy="most_frequent").fit(Xtr, ytr).predict(Xte)))
    if has_gender:
        for g in sorted(Xte["Gender"].unique()):
            m = (Xte["Gender"].values == g) & (yte == drop_idx)
            if m.sum() > 0:
                sub_recalls.setdefault(g, []).append(float((pred[m] == drop_idx).mean()))

ms = lambda a: f"{np.mean(a):.3f} ± {np.std(a, ddof=1):.3f}"
recalls = np.array(recalls)
per_class_txt = ", ".join(f"{c}: {ms(recalls[:, i])}" for i, c in enumerate(CLASSES))
sub_txt = ("; ".join(f"Dropout recall for Gender={g}: {ms(v)} (n_seeds={len(v)})"
                     for g, v in sub_recalls.items())
           if has_gender else "Gender column not found — subgroup analysis NOT run.")

print("Accuracy      :", ms(accs))
print("Macro-F1      :", ms(f1s))
print("Majority-class accuracy (referencia):", ms(majs))
print("Recall por clase:", per_class_txt)
print(sub_txt)

Accuracy      : 0.777 ± 0.010
Macro-F1      : 0.709 ± 0.015
Majority-class accuracy (referencia): 0.499 ± 0.000
Recall por clase: Dropout: 0.763 ± 0.023, Enrolled: 0.423 ± 0.029, Graduate: 0.914 ± 0.008
Dropout recall for Gender=0: 0.737 ± 0.014 (n_seeds=5); Dropout recall for Gender=1: 0.791 ± 0.047 (n_seeds=5)


## Ejercicio 4 · Model Card (9 secciones)
Mismas claves y estructura que el cuaderno de la profesora (el scorecard cuenta exactamente 9 secciones). Los números vienen de la celda anterior.

In [9]:
model_card = {
    "Model details": {
        "Name / version": "StudentDropout-GBM v0.1",
        "Owner / contact": "FELIX RICHAR MENDOZA PUMACAHUA <felix.mendozap@unmsm.edu.pe>",
        "Date": date.today().isoformat(),
        "Architecture": "Gradient Boosting (200 trees, depth 3) on standardised features; "
                        f"scikit-learn {sklearn.__version__}, Python {platform.python_version()}",
        "License": "Code: MIT (proposed by the author). Data: CC BY 4.0 (UCI).",
    },
    "Intended use": {
        "Primary intended uses": "Course exercise (Research Methods, UNMSM) to practise documentation "
                                 "of a student-dropout classifier. Research on early-warning methods.",
        "Out-of-scope / misuse": "NOT for decisions about real students (admission, sanctions, funding, "
                                 "support allocation). Trained on one institution's historical records; "
                                 "no fairness audit and no external validation.",
        "Intended users": "Doctoral students in this course; reviewers of this exercise.",
    },
    "Factors": {
        "Relevant groups / conditions": "Dataset includes demographic and socioeconomic attributes "
                                        "(e.g., marital status, nationality, parents' qualifications"
                                        + (", gender" if has_gender else "") + ") and three macroeconomic "
                                        "indicators (unemployment, inflation, GDP). Degree programme (17 degrees) and "
                                        "cohort (2008/09–2018/19) are also predictors, so programme and cohort effects "
                                        "are entangled with the other factors.",
        "Evaluation factors": "Reported overall, per class and by Gender (Dropout recall)." if has_gender
                              else "Reported overall and per class.",
    },
    "Metrics": {
        "Performance measures": "Accuracy and macro-F1 (classes are imbalanced, so accuracy alone misleads); "
                                "per-class recall.",
        "Decision thresholds": "argmax of predicted class probabilities (no threshold tuning).",
        "Variation approach": f"Mean ± std (ddof=1) over {len(list(SEEDS))} random seeds; each seed changes "
                              "both the stratified split and the model's random_state.",
    },
    "Evaluation data": {
        "Datasets": f"{int(TEST_SIZE*100)}% stratified held-out split of UCI #697 ({N} students), "
                    "re-drawn for each seed.",
        "Preprocessing": "StandardScaler fit on the training split only (no leakage into test).",
    },
    "Training data": {
        "Datasets": f"{int((1-TEST_SIZE)*100)}% of UCI #697 ({N} students, {P} features, "
                    f"classes: {class_counts_txt}).",
        "Known limitations": "Single institution (Polytechnic Institute of Portalegre), historical, "
                             "class imbalance (Graduate 50% / Dropout 32% / Enrolled 18% per Realinho et al., 2022). "
                             f"{len(sem_cols)} features describe curricular units of the 1st and 2nd semesters, so the "
                             "model is NOT an at-enrolment predictor: it can only be applied after those semesters. "
                             "The dataset authors also report pairs of predictors with Pearson r > 0.7 "
                             "(multicollinearity), which complicates interpretation of feature importance.",
    },
    "Quantitative analyses": {
        "Overall results": f"accuracy={ms(accs)}, macro-F1={ms(f1s)} "
                           f"(majority-class accuracy for reference: {ms(majs)})",
        "Disaggregated results": f"per-class recall: {per_class_txt}. {sub_txt}",
    },
    "Ethical considerations": {
        "Risks": "Labelling students as 'likely to drop out' can become self-fulfilling or stigmatising; "
                 "protected/sensitive attributes may encode structural inequality; false negatives leave "
                 "at-risk students unsupported.",
        "Mitigations": "Educational use only; no deployment; any real use would need a fairness audit, "
                       "human review and consent/legal review (UNKNOWN — to investigate).",
    },
    "Caveats and recommendations": {
        "Caveats": "Results will NOT transfer to other institutions, countries or cohorts. "
                   "Single train/test protocol, no hyper-parameter tuning, no baseline beyond majority class.",
        "Recommendations": "External validation, calibration check, subgroup fairness audit and a tuned "
                           "baseline (e.g., regularised logistic regression) before any claim of superiority. "
                           "Dataset authors suggest imbalance-aware methods (e.g., SMOTE, ADASYN, balanced bagging); "
                           "not applied here.",
    },
}

def render_card(card, title):
    lines = [f"# {title}", ""]
    for section, fields in card.items():
        lines.append(f"## {section}")
        for k, v in fields.items():
            lines.append(f"- **{k}:** {v}")
        lines.append("")
    return "\n".join(lines)

card_md = render_card(model_card, "Model Card — Student Dropout Classifier (educational use)")
print(card_md)
with open("model_card.md", "w", encoding="utf-8") as f:
    f.write(card_md)
print("\n✅ Saved model_card.md")

# Model Card — Student Dropout Classifier (educational use)

## Model details
- **Name / version:** StudentDropout-GBM v0.1
- **Owner / contact:** FELIX RICHAR MENDOZA PUMACAHUA <felix.mendozap@unmsm.edu.pe>
- **Date:** 2026-10-23
- **Architecture:** Gradient Boosting (200 trees, depth 3) on standardised features; scikit-learn 1.6.1, Python 3.13.16
- **License:** Code: MIT (proposed by the author). Data: CC BY 4.0 (UCI).

## Intended use
- **Primary intended uses:** Course exercise (Research Methods, UNMSM) to practise documentation of a student-dropout classifier. Research on early-warning methods.
- **Out-of-scope / misuse:** NOT for decisions about real students (admission, sanctions, funding, support allocation). Trained on one institution's historical records; no fairness audit and no external validation.
- **Intended users:** Doctoral students in this course; reviewers of this exercise.

## Factors
- **Relevant groups / conditions:** Dataset includes demographic and socioeconomic

## Ejercicio 5 · Datasheet (7 secciones)
Lo tomado de Realinho et al. (2022) y de la página de UCI está escrito como hecho; lo que esas fuentes no dicen queda como `UNKNOWN — to investigate`.

In [10]:
datasheet = {
    "Motivation": {
        "Purpose": "Built to train machine-learning models that estimate each student's risk of dropping out or "
                   "failing; the models feed a Learning Analytics tool that helps tutors target support "
                   "(Realinho et al., 2022; Martins et al., 2021).",
        "Created by / funded by": "Valentim Realinho, Jorge Machado, Luís Baptista, Mónica V. Martins — "
                                  "Polytechnic Institute of Portalegre (Portugal). Funding (per UCI): programme SATDAP "
                                  "(Capacitação da Administração Pública), grant POCI-05-5762-FSE-000191.",
    },
    "Composition": {
        "What instances represent": f"One row per student: {N} students, {P} predictors in the downloaded table. "
                                    "Predictor groups per the paper: demographic, socioeconomic, macroeconomic "
                                    "(unemployment, inflation, GDP), academic at enrolment, and academic performance "
                                    "at the end of the 1st and 2nd semesters. "
                                    "Attribute count: the paper reports 35 attributes (34 predictors + target) while UCI lists 36 "
                                    f"features (UNKNOWN — to investigate the difference). Cross-check run: {paper_check_txt}.",
        "Labels": "Dropout / Enrolled / Graduate, defined by the student's status at the end of the normal course "
                  "duration (3 years; 4 for Nursing). Dropout is defined broadly: a change of course or institution "
                  f"counts as dropout. Counts in the downloaded table: {class_counts_txt} (imbalanced).",
        "Sample vs population": "Students of 17 undergraduate degrees at ONE polytechnic institute, enrolled 2008/09–2018/19; "
                                "NOT representative of other institutions or countries. Outcome rates vary strongly by degree.",
        "Missing data": f"None in the downloaded table ({int(X.isna().sum().sum())} missing values). The paper reports "
                        "'rigorous preprocessing' of missing values; the method used (e.g., imputation) is UNKNOWN — to investigate.",
    },
    "Collection process": {
        "How collected": "Merged from four sources: the institution's Academic Management System (raw export of "
                         "13,992 rows × 398 columns), its Support System for the Teaching Activity (PAE), national "
                         "admission data (DGES, via CNAES), and macroeconomic data from PORDATA entered manually "
                         "(Realinho et al., 2022).",
        "Timeframe": "Academic years 2008/2009 to 2018/2019 (after the Bologna process); donated to UCI on 2021-12-12.",
        "Consent / ethics": "Data are anonymised; privacy aspects were validated by the institution's Data Protection "
                            "Officer under GDPR. The paper lists informed consent as 'not applicable'. "
                            "Ethics-committee approval: UNKNOWN — to investigate.",
    },
    "Preprocessing / cleaning / labeling": {
        "Steps": "Creators removed records from discontinued courses and Erasmus enrolments, removed duplicates, and handled "
                 "anomalies, outliers and missing values (details UNKNOWN — to investigate). Labelling per the outcome rule above. "
                 "We additionally apply StandardScaler at train time (fit on the train split only).",
        "Raw data available": "Only the processed table is public (UCI, Zenodo). Availability of the raw AMS/PAE/DGES "
                              "extracts: UNKNOWN — to investigate (the paper does not say they are released).",
    },
    "Uses": {
        "Appropriate uses": "Comparative studies of student academic performance, benchmarking and teaching of machine "
                            "learning (uses suggested by the dataset authors), and early-warning research with a fairness review.",
        "Should NOT be used for": "Decisions about individual students (admission, sanctions, funding) without a fairness "
                                  "audit and human review; generalising to other institutions or countries; "
                                  "prediction at enrolment time (12 predictors only exist after the 1st/2nd semesters).",
    },
    "Distribution": {
        "How distributed": "Public via the UCI Machine Learning Repository (ID 697, DOI 10.24432/C5MC89) and Zenodo "
                           "(doi 10.5281/zenodo.5777339). Dataset paper: Realinho et al., Data 7(11):146, DOI 10.3390/data7110146.",
        "License / restrictions": "Creative Commons Attribution 4.0 (CC BY 4.0): attribution required.",
    },
    "Maintenance": {
        "Maintainer": "Hosted by UCI and Zenodo; the creators' own maintenance commitment is UNKNOWN — to investigate. "
                      "The institution uses the derived models in an internal Learning Analytics tool.",
        "Update plan": "None stated on the dataset page or in the paper.",
    },
}

sheet_md = render_card(datasheet, "Datasheet — UCI #697 Predict Students' Dropout and Academic Success (educational use)")
print(sheet_md)
with open("datasheet.md", "w", encoding="utf-8") as f:
    f.write(sheet_md)
print("\n✅ Saved datasheet.md")

# Datasheet — UCI #697 Predict Students' Dropout and Academic Success (educational use)

## Motivation
- **Purpose:** Built to train machine-learning models that estimate each student's risk of dropping out or failing; the models feed a Learning Analytics tool that helps tutors target support (Realinho et al., 2022; Martins et al., 2021).
- **Created by / funded by:** Valentim Realinho, Jorge Machado, Luís Baptista, Mónica V. Martins — Polytechnic Institute of Portalegre (Portugal). Funding (per UCI): programme SATDAP (Capacitação da Administração Pública), grant POCI-05-5762-FSE-000191.

## Composition
- **What instances represent:** One row per student: 4424 students, 36 predictors in the downloaded table. Predictor groups per the paper: demographic, socioeconomic, macroeconomic (unemployment, inflation, GDP), academic at enrolment, and academic performance at the end of the 1st and 2nd semesters. Attribute count: the paper reports 35 attributes (34 predictors + target) while UCI lis

## Ejercicio 6 · Scorecard (autoauditoría)
Copia exacta de la función de la profesora. El objetivo es **8/8**.

In [7]:
def score_documentation(model_card, datasheet):
    checks = []
    def add(label, passed, why=""):
        checks.append((label, bool(passed), why))
    def flatten(card):
        parts = []
        for fields in card.values():
            for k, v in fields.items():
                parts.append(str(k).lower()); parts.append(str(v).lower())
        return " ".join(parts)
    flat_mc = flatten(model_card); flat_ds = flatten(datasheet)
    add("Model Card has all 9 sections", len(model_card) == 9)
    add("Datasheet has all 7 sections", len(datasheet) == 7)
    add("Intended MISUSE / out-of-scope stated",
        ("not for" in flat_mc) or ("out-of-scope" in flat_mc) or ("should not" in flat_mc))
    add("Performance reported with variation (± / std / seeds)",
        ("±" in flat_mc) or ("std" in flat_mc) or ("seed" in flat_mc))
    add("Metrics disaggregated (per-class / per-group)",
        ("per-class" in flat_mc) or ("per class" in flat_mc) or ("disaggreg" in flat_mc))
    add("Dataset 'should NOT be used for' stated",
        ("should not be used" in flat_ds) or ("not for" in flat_ds))
    add("Collection process described", len(datasheet["Collection process"]) > 0
        and "how collected" in (" ".join(datasheet["Collection process"].keys()).lower()))
    add("Gaps flagged honestly ('unknown')", ("unknown" in flat_mc) or ("unknown" in flat_ds),
        "Flagging what you don't know beats a confident blank.")
    passed = sum(1 for _, p, _ in checks if p)
    print("DOCUMENTATION SCORECARD"); print("=" * 52)
    for label, p, why in checks:
        print("[" + ("✓" if p else "✗") + "] " + label)
        if why and not p: print("      hint: " + why)
    print("=" * 52); print(f"Score: {passed}/{len(checks)}  ({100*passed/len(checks):.0f}%)")
    return passed, len(checks)

_ = score_documentation(model_card, datasheet)

DOCUMENTATION SCORECARD
[✓] Model Card has all 9 sections
[✓] Datasheet has all 7 sections
[✓] Intended MISUSE / out-of-scope stated
[✓] Performance reported with variation (± / std / seeds)
[✓] Metrics disaggregated (per-class / per-group)
[✓] Dataset 'should NOT be used for' stated
[✓] Collection process described
[✓] Gaps flagged honestly ('unknown')
Score: 8/8  (100%)


## Checklist final antes de entregar
- [ ] Leí Realinho et al. (2022) resumen y verifiqué datos del Datasheet (fuentes, periodo, 17 carreras, consentimiento, preprocesamiento).
- [ ] Revisé la salida de «Contraste con el artículo» (registros, clases, atributos) y anoté cualquier diferencia.
- [ ] Confirmé que el uso previsto es *después* del 1.º/2.º semestre (12 predictores son de rendimiento semestral).
- [ ] Scorecard 8/8.
- [ ] Subí `model_card.md` y `datasheet.md` al GitHub (carpeta del entregable 7) y/o a la plataforma del curso.

**Referencias:** Mitchell et al. (2019), *Model Cards for Model Reporting*, arXiv:1810.03993 · Gebru et al. (2021), *Datasheets for Datasets*, CACM 64(12) · Realinho, Machado, Baptista y Martins (2022), *Predicting Student Dropout and Academic Success*, Data 7(11):146, DOI 10.3390/data7110146 · UCI ML Repository #697, DOI 10.24432/C5MC89 · Martins et al. (2021), *Early prediction of student's performance in higher education: a case study*, WorldCIST 2021.